# Contrastive learning (SimCLR, CLIP)

_Modern Deep Learning & AI_

**Pull two views of the same thing together, push everything else apart. No labels needed.**

Labels are expensive. Self-supervised learning learns useful features from raw data with no human labels.

     Contrastive learning is one way. Take an image, make two altered copies (crop, recolour). These are a positive pair: they should land close together in feature space.

---

This notebook is a practice scaffold for the **Contrastive learning (SimCLR, CLIP)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We build the **NT-Xent / InfoNCE** loss that powers SimCLR in three steps: (1) define the loss that pulls positive pairs together and pushes everything else apart, (2) make a synthetic batch with two views per item, (3) score it. The key idea: each item's two augmented views are a *positive pair*; every other item in the batch is a *negative*.

### Step 1 — Define the NT-Xent contrastive loss

We stack both views into one `2B` block and L2-normalize them, so a dot product becomes a **cosine similarity**. Dividing by the temperature `tau` sharpens the distribution. We blank the diagonal so nothing matches itself, then point each row at its true partner (B positions away) and let cross-entropy do the softmax — that *is* InfoNCE.

In [ ]:
import torch
import torch.nn.functional as F

def nt_xent(z1, z2, tau=0.1):
    # z1, z2: (B, d) embeddings of the two views of the same B inputs
    B = z1.size(0)

    stacked = torch.cat([z1, z2], dim=0)         # (2B, d): both views together
    z = F.normalize(stacked, dim=1)             # (2B, d) unit vectors

    sim = (z @ z.t()) / tau                       # cosine sim matrix (2B, 2B)
    sim.fill_diagonal_(float('-inf'))             # no self-comparison

    # positive of row i is its partner B positions away (and vice versa)
    first_half = torch.arange(B, 2 * B)           # rows 0..B-1 point to their view-2 partner
    second_half = torch.arange(0, B)              # rows B..2B-1 point back to view-1
    targets = torch.cat([first_half, second_half])

    return F.cross_entropy(sim, targets)          # InfoNCE = softmax CE

### Step 2 — Build a synthetic batch with two views

We fake 4 items as 8-dim embeddings (`z1`), then make a second view (`z2`) by adding a tiny perturbation. Because the two views differ only slightly, an item's two views should be far more similar to each other than to any other item — exactly the structure the loss rewards.

In [ ]:
# synthetic batch: 4 items, 8-dim embeddings, two views each
z1 = torch.randn(4, 8)
z2 = z1 + 0.05 * torch.randn(4, 8)   # view 2 is a slightly perturbed view 1

### Step 3 — Compute the loss

We run the batch through `nt_xent`. Because the positive pairs sit close and the negatives are random, the loss should come out low — the network would be rewarded for keeping each pair's two views together.

In [ ]:
loss = nt_xent(z1, z2, tau=0.1)
print(float(loss))

## Visualize the data & results

_On real handwritten-digit images, does the cosine-similarity matrix light up where two samples of the same digit meet?_

We do this in three steps: (1) pick two real images of each of four digit classes — these are our positive pairs, (2) L2-normalize them and compute the full cosine-similarity matrix, (3) draw the matrix and check whether same-digit cells glow warm.

### Step 1 — Pick two real samples of each digit

For digits 0, 1, 2, 3 we grab two different real images each (`a` and `b`). Each `(a, b)` pair for the same digit is a **positive pair** — the contrastive analogue of two augmented views — so we expect those two to be the most similar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()                # real 8x8 handwritten digit images
X = digits.data
y = digits.target

# Two real samples of each digit class 0,1,2,3 = the positive pairs.
rows = []
names = []
for tag, k in [('a', 0), ('b', 1)]:
    for d in [0, 1, 2, 3]:
        sample = X[y == d][k]         # the k-th real image of digit d
        rows.append(sample)
        names.append(f'd{d}_{tag}')

### Step 2 — Normalize and build the cosine-similarity matrix

We stack the eight images, L2-normalize each row to unit length, then take `Z @ Z.T`. Because the rows are unit vectors, that dot product is exactly the **cosine similarity** between every pair of images.

In [ ]:
Z = np.array(rows, dtype=float)
norms = np.linalg.norm(Z, axis=1, keepdims=True)
Z = Z / (norms + 1e-9)                # L2-normalize each row to unit length
S = Z @ Z.T                           # cosine similarity between all pairs

### Step 3 — Draw the similarity matrix

We render `S` as a heatmap with the numeric value in each cell. Watch for warm cells linking the two samples of the same digit (e.g. `d0_a` with `d0_b`) — that bright off-diagonal is the signal contrastive learning is built to amplify.

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(S, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='cosine similarity')
plt.xticks(range(8), names, rotation=45)
plt.yticks(range(8), names)
plt.title('Cosine similarity of real digit embeddings')

# annotate each cell with its similarity value
for i in range(8):
    for j in range(8):
        plt.text(j, i, format(S[i, j], '.2f'), ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()